In [1]:
"""
01_parse_plates.ipynb
===========================
Parses raw Harmony PlateResults exports and merges with plate map metadata
to produce annotated well-level feature CSVs, then stacks into a combined profile.

Inputs:
    - data/raw/plate_results/    PlateResults_[date]_[plate].txt files exported from Harmony
    - data/raw/plate_maps/       PlateMaps_[date].xlsx files with sheets named by plate ID

Outputs:
    - data/processed/            PlateData_[date]_[plate].csv (one per plate, annotated)
    - data/processed/all_profiles.csv (combined profile across all plates, MT/MT excluded)
"""

import pandas as pd
from pathlib import Path

In [2]:
# Define paths
RAW_DIR = Path("../data/raw")
PLATE_RESULTS_DIR = RAW_DIR / "plate_results"
PLATE_MAPS_DIR = RAW_DIR / "plate_maps"
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(exist_ok=True)

In [3]:
# Helper functions (will be called in workflow)
def parse_plate_results(txt_path):
    """Read a PlateResults txt, extract plate name and feature data."""
    plate_name = None
    data_start = None
    
    with open(txt_path, "r") as f:
        lines = f.readlines()
    
    # Extract plate name and find [Data] section
    for i, line in enumerate(lines):
        if line.startswith("Plate Name"):
            plate_name = line.split("\t")[1].strip()
        if line.strip() == "[Data]":
            data_start = i + 1
            break
    
    # Read data section
    from io import StringIO
    data_str = "".join(lines[data_start:])
    df = pd.read_csv(StringIO(data_str), sep="\t")
    df.insert(0, "plate", plate_name)

    # Dropping all NA columns (Compound, Concentraton, Cell Type, Cell Count)
    df = df.dropna(axis=1, how="all")
    
    return df, plate_name

def parse_plate_map(date, plate):
    """Read the plate map sheet for a given date and plate."""
    map_path = PLATE_MAPS_DIR / f"PlateMaps_{date}.xlsx"
    df = pd.read_excel(map_path, sheet_name=plate)
    # Standardize column names just in case
    df.columns = df.columns.str.strip().str.lower()
    df = df.rename(columns={"row": "Row", "col": "Column", "dose_um": "dose_uM"})
    return df

def get_condition(row):
    """Derive condition label from cell_type and compound."""
    cell_type = str(row["cell_type"]).strip()
    compound = str(row["compound"]).strip().upper()
    is_neg = compound == "NEG CTRL"
    
    if cell_type == "WT/WT" and is_neg:
        return "healthy"
    elif cell_type == "WT/MT" and is_neg:
        return "disease"
    elif cell_type == "MT/MT" and is_neg:
        return "disease_hom" 
    elif cell_type == "WT/MT" and not is_neg:
        return "treatment"
    elif cell_type == "MT/MT" and not is_neg:
        return "treatment_hom"
    else:
        return None

In [4]:
print("="*60)
print("1. PROCESS ALL PLATE RESULT TXT FILES WITH PLATE MAP METADATA")
print("="*60)
print()

plate_files = sorted(PLATE_RESULTS_DIR.glob("PlateResults_*.txt"))
print(f"Found {len(plate_files)} plate result files...\n")

for txt_path in plate_files:
    # Parse filename to get date and plate
    stem = txt_path.stem  # expects following naming convention: PlateResults_[Date]_[Plate] e.g. PlateResults_20250609_A1
    parts = stem.split("_")
    date = parts[1]
    plate = parts[2]
    
    print(f"Processing {txt_path.name}...")
    
    # Parse plate results file
    df, plate_name = parse_plate_results(txt_path)
    df.insert(1, "session", date)
    
    # Parse plate map file
    plate_map = parse_plate_map(date, plate)

    # Merge plate results + plate map on Row and Column
    df = df.merge(plate_map, on=["Row", "Column"], how="left")
    
    # Reorder: plate_name, Row, Column, compound, dose_uM, cell_type, then rest
    meta_cols = ["plate", "session", "Row", "Column", "compound", "dose_uM", "cell_type"]
    feature_cols = [c for c in df.columns if c not in meta_cols]
    df = df[meta_cols + feature_cols]

    # Add "condition" column right after cell_type
    df.insert(df.columns.get_loc("cell_type") + 1, "condition", df.apply(get_condition, axis=1))
    
    # Save to csv
    out_path = PROCESSED_DIR / f"PlateData_{date}_{plate}.csv"
    df.to_csv(out_path, index=False)
    print(f"  Saved to {out_path.name} — {len(df)} wells, {len(df.columns)} columns")

print("\nDone.")

1. PROCESS ALL PLATE RESULT TXT FILES WITH PLATE MAP METADATA

Found 14 plate result files...

Processing PlateResults_20250609_A1.txt...
  Saved to PlateData_20250609_A1.csv — 60 wells, 2145 columns
Processing PlateResults_20250609_A2.txt...
  Saved to PlateData_20250609_A2.csv — 60 wells, 2145 columns
Processing PlateResults_20250609_B1.txt...
  Saved to PlateData_20250609_B1.csv — 60 wells, 2145 columns
Processing PlateResults_20250609_B2.txt...
  Saved to PlateData_20250609_B2.csv — 60 wells, 2145 columns
Processing PlateResults_20250616_A1.txt...
  Saved to PlateData_20250616_A1.csv — 60 wells, 2145 columns
Processing PlateResults_20250616_A2.txt...
  Saved to PlateData_20250616_A2.csv — 60 wells, 2145 columns
Processing PlateResults_20250616_B1.txt...
  Saved to PlateData_20250616_B1.csv — 60 wells, 2145 columns
Processing PlateResults_20250616_B2.txt...
  Saved to PlateData_20250616_B2.csv — 60 wells, 2145 columns
Processing PlateResults_20250623_A1.txt...
  Saved to PlateData_2

In [5]:
print("="*60)
print("2. COMBINE AND SAVE ALL PLATE DATA TO ALL_PROFILES.CSV")
print("="*60)
print()

all_plates = []

for csv_path in sorted(PROCESSED_DIR.glob("PlateData_*.csv")):
    plate_df = pd.read_csv(csv_path)
    all_plates.append(plate_df)

master = pd.concat(all_plates, ignore_index=True)

# Filter out MT/MT (homozygous mutant) — not biologically relevant
mt_hom_count = (master["cell_type"] == "MT/MT").sum()
master = master[master["cell_type"] != "MT/MT"]
print(f"{mt_hom_count} MT/MT rows filtered out")
assert master["condition"].isna().sum() == 0, f"Unexpected NaN conditions: {master[master['condition'].isna()][['Row','Column','cell_type','compound']]}"

# Save to csv
master_path = PROCESSED_DIR / "all_profiles.csv"
master.to_csv(master_path, index=False)
print(f"\nMaster data saved to {master_path.name} — {len(master)} wells, {len(master.columns)} columns")
print(f"\nCondition counts:\n{master['condition'].value_counts()}")

2. COMBINE AND SAVE ALL PLATE DATA TO ALL_PROFILES.CSV

324 MT/MT rows filtered out

Master data saved to all_profiles.csv — 504 wells, 2145 columns

Condition counts:
condition
treatment    288
disease      108
healthy      108
Name: count, dtype: int64
